# Gold Layer (Reference)
Completed version — delete this once I've built my own.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date, timedelta

CATALOG_SCHEMA = "dev.electroflow_pipeline."

silver_customers_table = CATALOG_SCHEMA + "silver_customers"
silver_products_table = CATALOG_SCHEMA + "silver_products"
silver_orders_table = CATALOG_SCHEMA + "silver_orders"
silver_order_items_table = CATALOG_SCHEMA + "silver_order_items"

gold_dim_customers = CATALOG_SCHEMA + "dim_customers"
gold_dim_products = CATALOG_SCHEMA + "dim_products"
gold_dim_date = CATALOG_SCHEMA + "dim_date"
gold_fact_order_items = CATALOG_SCHEMA + "fact_order_items"

In [0]:
def create_dim_customers():
    df = spark.table(silver_customers_table)

    df_dim = df.select(
        "customer_id", "first_name", "last_name",
        "gender", "city", "zip_code", "country", "join_date"
    )

    df_dim.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_dim_customers)
    print("dim_customers done")
    return df_dim

In [0]:
def create_dim_products():
    df = spark.table(silver_products_table)

    df_dim = df.select(
        "product_id", "product_name", "brand", "category", "price"
    )

    df_dim.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_dim_products)
    print("dim_products done")
    return df_dim

In [0]:
def create_dim_date():
    start_date = date(2024, 1, 1)
    end_date = date(2026, 12, 31)
    date_list = [{"date": start_date + timedelta(days=i)} for i in range((end_date - start_date).days + 1)]

    df = spark.createDataFrame(date_list)

    df_dim = df \
        .withColumn("date_id", F.date_format(F.col("date"), "yyyy-MM-dd")) \
        .withColumn("year", F.year(F.col("date"))) \
        .withColumn("quarter", F.quarter(F.col("date"))) \
        .withColumn("month", F.month(F.col("date"))) \
        .withColumn("day_of_week", F.dayofweek(F.col("date"))) \
        .withColumn("is_weekend", F.when(F.dayofweek(F.col("date")).isin(1, 7), True).otherwise(False)) \
        .drop("date")

    df_dim.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_dim_date)
    print("dim_date done")
    return df_dim

In [0]:
def create_fact_order_items():
    df_items = spark.table(silver_order_items_table)
    df_orders = spark.table(silver_orders_table)
    df_products = spark.table(silver_products_table)

    # join items with orders to get customer_id and timestamp
    df_joined = df_items.join(
        df_orders.select("order_id", "customer_id", "order_purchase_timestamp"),
        on="order_id",
        how="inner"
    )

    # join with products to get the unit price
    df_joined = df_joined.join(
        df_products.select("product_id", F.col("price").alias("unit_price")),
        on="product_id",
        how="inner"
    )

    # calculate revenue
    df_fact = df_joined \
        .withColumn("date_id", F.date_format(F.col("order_purchase_timestamp"), "yyyy-MM-dd")) \
        .withColumn("item_revenue", (F.col("quantity") * F.col("unit_price")).cast("decimal(18,2)"))

    df_fact = df_fact.select(
        "order_id", "product_id", "customer_id",
        "date_id", "quantity", "unit_price", "item_revenue"
    )

    df_fact.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(gold_fact_order_items)
    print("fact_order_items done")
    return df_fact

In [0]:
create_dim_customers()
create_dim_products()
create_dim_date()
create_fact_order_items()
print("gold layer complete")